In [ ]:
%cd /kaggle/working/
!rm -rf /kaggle/working/SimVLA
!git clone https://github.com/nguyenduchuyiu/vla-so101 SimVLA

In [ ]:
!pip install -q av

In [ ]:
from pathlib import Path

# ===== Paths cần sửa =====
REPO_DIR = Path('/kaggle/working/SimVLA')
DATA_SOURCE = Path('/kaggle/input/datasets/nguyenduchuhu/so101-data/so101_counterfactual_observable')
MODEL_SOURCE = 'HuggingFaceTB/SmolVLM-500M-Instruct'  # hoặc path model trong /kaggle/input
OUTPUT_DIR = Path('/kaggle/working/runs/so101_full_unfrozen')

# ===== Train =====
INSTALL_DEPS = True
BATCH_PER_GPU = 128       
ITERS = 4_000
SAVE_INTERVAL = 1_000
LEARNING_RATE = 5e-5
LEARNING_COEF = 1.0     # VLM LR = learning_rate * learning_coef
RESUME_CKPT = '/kaggle/input/notebooks/nguyenduchuhu/vla-so101/runs/so101_full_unfrozen/ckpt-1000/model.safetensors'        # để trống nếu train mới

# ===== Rollout test =====
TEST_SEED = 76
SOURCE_INDEX = 0        # red cube
TARGET_INDEX = 0        # green tray
INSTRUCTION = 'Pick up the small red cube and place it on the green tray.'
POLICY_SEED = 0
EXECUTE_STEPS = 1
MAX_REPLANS = 220


In [ ]:
import os
import subprocess
import sys

if INSTALL_DEPS:
    packages = [
        'transformers==4.57.6', 'accelerate', 'safetensors', 'tensorboard',
        'fastapi', 'uvicorn', 'json-numpy', 'mmengine', 'mediapy',
        'num2words', 'opencv-python-headless', 'so101-nexus==0.4.8', 'peft==0.17.1',
    ]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)

os.environ['PYTHONPATH'] = str(REPO_DIR.parent / 'src')
os.environ['MUJOCO_GL'] = 'egl'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['WANDB_DISABLED'] = 'true'
os.chdir(REPO_DIR)

assert REPO_DIR.is_dir(), f'Repo không tồn tại: {REPO_DIR}'
assert (DATA_SOURCE / 'meta' / 'episodes.jsonl').is_file(), f'Data source sai: {DATA_SOURCE}'

import torch
assert torch.cuda.device_count() == 2, f'Cần bật Kaggle GPU T4 x2, hiện có {torch.cuda.device_count()} GPU'
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(2)])
print('Data:', DATA_SOURCE)

In [ ]:
TRAIN_META = REPO_DIR / 'datasets/metas/so101_observable_train.json'
NORM_STATS = REPO_DIR / 'norm_stats/so101_observable_norm.json'

train_env = os.environ.copy()
train_env.update({
    'SO101_DATA_DIR': str(DATA_SOURCE),
    'TRAIN_METAS_PATH': str(TRAIN_META),
    'NORM_STATS_PATH': str(NORM_STATS),
    'SMOLVLM_MODEL_PATH': str(MODEL_SOURCE),
    'REBUILD_DATA': '1',
    'ITERS': str(ITERS),
    'SAVE_INTERVAL': str(SAVE_INTERVAL),
    'LEARNING_RATE': str(LEARNING_RATE),
    'LEARNING_COEF': str(LEARNING_COEF),
    'PYTHONUNBUFFERED': '1',
})
train_cmd = [
    'bash',f'{REPO_DIR}/train_smolvlm_so101_kaggle.sh',
    str(BATCH_PER_GPU), str(OUTPUT_DIR),
]
if RESUME_CKPT:
    train_cmd.append(str(RESUME_CKPT))

print('Running:', ' '.join(train_cmd))
subprocess.run(train_cmd, env=train_env, check=True)